## Base v22 — Adicionar autor à base (Edgar Allan Poe)

Regras de versionamento (sempre):
- **Nunca sobrescrever** `df_full_v*.pq` / `df_full_encoded_v*.pq` existentes — cada alteração de base gera a versão seguinte (v21→v22).
- Schema das colunas idêntico à v21 (`title, author, extension, class, subtitle, path_raw, path_txt, text_raw_len, text, text_len, text_clean, text_clean_len, weights, split` + `text_encoded, text_encoded_len`).
- Fluxo espelhado do `playground_2.ipynb` (adição do Tolkien): epub/pdf → txt → `clean_text2` → concat com base atual → weights → split → encode com o tokenizer **existente** (`gpt2_ptbr_50k_v2`, sem retreinar).

**Fontes do Poe (BR, já em `data/poe/`)**: Navras Digital vol 1 (pdf) e vol 2 (epub), Todos Os Contos (pdf), DarkSide **Medo Clássico vol 1** (epub), Aleph **Contos de Ficção Científica** 2025 (epub).

In [1]:
# CONFIG — troque o AUTOR e a pasta de brutos para adicionar outro autor
AUTOR = "chambers"
RAW_DIR = f"data/{AUTOR}"          # .epub/.pdf brutos
TXT_DIR = f"data/{AUTOR}_txt"      # .txt convertidos (gerado aqui)
BASE_TXT = "data/df_full_v21.pq"   # base ATUAL (texto+clean) — não mexer
BASE_ENC = "data/df_full_encoded_v21.pq"  # base ATUAL (encoded) — referência p/ stats
TOKENIZER = "artifacts/tokenizers/gpt2_ptbr_50k_v2"
WEIGHT_CLIP = 500_000
SPLIT_FRAC = 0.85
RANDOM_STATE = 1
import os
Path = __import__('pathlib').Path
Path(TXT_DIR).mkdir(parents=True, exist_ok=True)
print(RAW_DIR, "existe:", os.path.isdir(RAW_DIR))

data/chambers existe: True


In [2]:
import glob, re
vers = sorted(int(re.search(r'df_full_v(\d+)\.pq', p).group(1))
              for p in glob.glob('data/df_full_v*.pq'))
BASE_TXT = f'data/df_full_v{max(vers)}.pq'          # base ATUAL = última versão
BASE_ENC = f'data/df_full_encoded_v{max(vers)}.pq'
NEXT_VERSION = max(vers) + 1
print(f'Bases encontradas: v{vers}')
print(f'base ATUAL: {BASE_TXT} | nova: v{NEXT_VERSION}')
OUT_TXT = f'data/df_full_v{NEXT_VERSION}.pq'
OUT_ENC = f'data/df_full_encoded_v{NEXT_VERSION}.pq'
print(f'saídas: {OUT_TXT} e {OUT_ENC}')

Bases encontradas: v[0, 21, 22]
base ATUAL: data/df_full_v22.pq | nova: v23
saídas: data/df_full_v23.pq e data/df_full_encoded_v23.pq


In [3]:
import ebooklib
from ebooklib import epub
from bs4 import BeautifulSoup

def sanitize_filename(name):
    # Windows: ':' '\\' '/' '*' '?' '"' '<' '>' '|' invalid -> substitui (evita NTFS ADS)
    return re.sub(r'[\\/:*?"<>|]', '_', name).strip()

def epub_to_txt(path_epub, folder_txt):
    book = epub.read_epub(path_epub)
    title = book.get_metadata('DC', 'title')[0][0]
    content = []
    for item in book.get_items():
        if item.get_type() == ebooklib.ITEM_DOCUMENT:
            soup = BeautifulSoup(item.get_content(), 'html.parser')
            content.append(soup.get_text())
    path_txt = folder_txt + "/" + sanitize_filename(title) + ".txt"
    with open(path_txt, 'w', encoding='utf-8') as f:
        f.write('\n'.join(content))
    return {'title': title,
            'author': book.get_metadata('DC', 'creator')[0][0] if book.get_metadata('DC', 'creator') else AUTOR,
            'path_raw': str(path_epub), 'path_txt': path_txt,
            'text_raw_len': len('\n'.join(content))}

epubs = [str(p) for p in Path(RAW_DIR).glob('*.epub')]
meta_epub = []
for f in epubs:
    try:
        m = epub_to_txt(f, TXT_DIR); meta_epub.append(m); print('  OK ', m['title'], '|', m['text_raw_len'])
    except Exception as e:
        print('  ERRO', f, e)

# dedupe por TÍTULO DC (mantém o com mais texto, descarta duplicata de scan/edição)
import pandas as pd
tmp = pd.DataFrame(meta_epub)
if len(tmp):
    tmp['_k'] = tmp.title.str.lower().str.strip()
    tmp['_rk'] = tmp.groupby('_k').text_raw_len.rank(method='first', ascending=False)
    dropped = tmp.loc[tmp._rk > 1, 'title'].tolist()
    if dropped:
        print('duplicatas descartadas por título:', dropped)
    meta_epub = tmp[tmp._rk == 1].drop(columns=['_k', '_rk']).to_dict('records')

In [4]:
import pdfplumber, re

def pdf_to_txt(pdf_path, txt_path):
    with pdfplumber.open(pdf_path) as pdf:
        text = "".join((pg.extract_text() or "") + "\n" for pg in pdf.pages)
    text = re.sub(r"\n{2,}", "\n\n", text)
    text = re.sub(r" +", " ", text).strip()
    with open(txt_path, 'w', encoding='utf-8') as f:
        f.write(text)
    return len(text)

pdfs = [str(p) for p in Path(RAW_DIR).glob('*.pdf')]
print('pdfs:', len(pdfs))
meta_pdf = []
for f in pdfs:
    title = Path(f).name.split(' - libgen.li')[0].replace('.pdf', '').strip()
    title = re.split(r' - ', title)[-1].strip()
    txt = f"{TXT_DIR}/{title}.pdf.txt"
    try:
        n = pdf_to_txt(f, txt)
        meta_pdf.append({'title': title, 'author': AUTOR, 'path_raw': f, 'path_txt': txt, 'text_raw_len': n})
        flag = '  <-- SUSPEITO (pouco texto; pdf pode ser scan)' if n < 20_000 else ''
        print('  OK ', title, '|', n, flag)
    except Exception as e:
        print('  ERRO', f, e)

pdfs: 1


  OK  O Rei de Amarelo (2015, Editora Clock Tower) | 488916 


In [5]:
import pandas as pd
df_new = pd.DataFrame(meta_epub + meta_pdf)
df_new['author'] = AUTOR
df_new['class'] = AUTOR
df_new['extension'] = df_new.path_raw.apply(lambda p: p.rsplit('.', 1)[-1])
df_new['subtitle'] = ""
df_new['text'] = df_new.path_txt.apply(lambda x: Path(x).read_text(encoding='utf-8'))
df_new['text_len'] = df_new['text'].str.len()

# Descarta docs sem texto (epub de quadrinhos/DRM = 0 bytes) e limpa o .txt vazio
n0 = (df_new['text_len'] == 0).sum()
if n0:
    print(f'AVISO: {n0} documento(s) sem texto (provável epub de quadrinhos/DRM). Descartando:')
    for t in df_new.loc[df_new.text_len == 0, 'title']:
        print('  -', t)
    for p in df_new.loc[df_new.text_len == 0, 'path_txt']:
        Path(p).unlink(missing_ok=True)
    df_new = df_new[df_new.text_len > 0].copy()
df_new = df_new.sort_values('text_raw_len', ascending=False).reset_index(drop=True)
print(df_new[['title','extension','text_raw_len']])

                                          title extension  text_raw_len
0  O Rei de Amarelo (2015, Editora Clock Tower)       pdf        488916


In [6]:
from src.prep import clean_text2
df_new['text_clean'] = df_new.text.apply(clean_text2)
df_new['text_clean_len'] = df_new['text_clean'].str.len()
print(df_new[['title','text_raw_len','text_clean_len']])

                                          title  text_raw_len  text_clean_len
0  O Rei de Amarelo (2015, Editora Clock Tower)        488916          488867


In [7]:
# Diagnóstico: marcadores BR x PT-PT (o tokenizer foi treinado em pt-BR moderno)
MARKS_BR  = ['você', 'fato,', 'direção', 'celular', 'a gente', 'não sei']
MARKS_PT  = ['facto,', 'direcção', 'telemóvel', 'connosco', 'comboio', 'tu ']
def variant_score(t):
    tl = ' ' + t.lower().replace('\n', ' ') + ' '
    return sum(tl.count(m) for m in MARKS_BR) - sum(tl.count(m) for m in MARKS_PT)
df_new['var_score'] = df_new.text_clean.apply(variant_score)
df_new['variante'] = df_new.var_score.apply(lambda s: 'BR' if s > 0 else ('PT-PT/arcaico?' if s < 0 else 'ambíguo'))
print(df_new[['title','variante','var_score']])
print('\nAVISO: docs com pouco texto (possível scan sem camada de texto):')
print(df_new[df_new.text_clean_len < 20_000][['title','text_clean_len']])

                                          title variante  var_score
0  O Rei de Amarelo (2015, Editora Clock Tower)       BR        569

AVISO: docs com pouco texto (possível scan sem camada de texto):
Empty DataFrame
Columns: [title, text_clean_len]
Index: []


In [8]:
cols_v21 = ['title','author','extension','class','subtitle','path_raw','path_txt',
            'text_raw_len','text','text_len','text_clean','text_clean_len','weights','split']
df_base = pd.read_parquet(BASE_TXT)
print('v21 colunas:', list(df_base.columns))
print('v21 autores:', df_base.author.value_counts().to_dict())
# GUARDA anti re-run: se o autor já está na base, não duplicar (versionamento vN+1 é p/ autor NOVO)
if AUTOR in set(df_base.author):
    raise SystemExit(f"'{AUTOR}' JÁ está na base atual ({BASE_TXT}). "
                     f"Para adicionar OUTRO autor, troque AUTOR na célula 1 (ex.: AUTOR='outro') "
                     f"e coloque os arquivos em data/<outro>/. Re-executar com AUTOR='{AUTOR}' criaria uma v{NEXT_VERSION} duplicada.")
df_base = pd.read_parquet(BASE_TXT)
print('v21 colunas:', list(df_base.columns))
print('v21 autores:', df_base.author.value_counts().to_dict())
assert set(cols_v21) <= set(df_base.columns), 'schema inesperado da v21'
# pesos/split são recalculados na célula seguinte; default temporário p/ concat
df_new['weights'] = 0.0
df_new['split'] = ''
df_full = pd.concat([df_base[cols_v21], df_new[cols_v21]], ignore_index=True)
# recálculo de len/clean p/ tudo (igual playground_2)
df_full['text_len'] = df_full['text'].str.len()
df_full['text_clean'] = df_full.text.apply(clean_text2)
df_full['text_clean_len'] = df_full['text_clean'].str.len()
print('total docs:', len(df_full), '| chars (clean) M:', round(df_full.text_clean_len.sum()/1e6, 2))

v21 colunas: ['title', 'author', 'extension', 'class', 'subtitle', 'path_raw', 'path_txt', 'text_raw_len', 'text', 'text_len', 'text_clean', 'text_clean_len', 'weights', 'split']
v21 autores: {'lovecraft': 114, 'king': 65, 'tolkien': 10, 'poe': 5}


v21 colunas: ['title', 'author', 'extension', 'class', 'subtitle', 'path_raw', 'path_txt', 'text_raw_len', 'text', 'text_len', 'text_clean', 'text_clean_len', 'weights', 'split']
v21 autores: {'lovecraft': 114, 'king': 65, 'tolkien': 10, 'poe': 5}


total docs: 195 | chars (clean) M: 71.47


In [9]:
df_full['weights'] = df_full['text_clean_len'].clip(0, WEIGHT_CLIP)
df_full['weights'] = df_full['weights'] / df_full['weights'].sum()
index_eval = df_full.sample(frac=1-SPLIT_FRAC, random_state=RANDOM_STATE, weights='weights').index
df_full['split'] = pd.Series(df_full.index.isin(index_eval)).map({False: 'train', True: 'eval'})
print(df_full.groupby(['author','split']).weights.sum())
print(df_full[['author','split']].value_counts().sort_index())

author     split
chambers   train    0.012500
king       eval     0.277111
           train    0.497949
lovecraft  eval     0.018322
           train    0.065833
poe        eval     0.012785
           train    0.029043
tolkien    eval     0.025043
           train    0.061414
Name: weights, dtype: float64
author     split
chambers   train      1
king       eval      22
           train     43
lovecraft  eval       3
           train    111
poe        eval       1
           train      4
tolkien    eval       3
           train      7
Name: count, dtype: int64


In [10]:
df_full = df_full.sample(frac=1).reset_index(drop=True)  # shuffle
df_full.to_parquet(OUT_TXT)
import os
print('salvo:', OUT_TXT, round(os.path.getsize(OUT_TXT)/1e6, 1), 'MB')
print('v21 intocada:', round(os.path.getsize(BASE_TXT)/1e6, 1), 'MB (não alterada)')
# tamanhos dos txt gerados p/ conferência
for t in sorted(Path(TXT_DIR).glob('*.txt')): print('  ', t.name, round(t.stat().st_size/1024), 'KB')

salvo: data/df_full_v23.pq 91.8 MB
v21 intocada: 91.1 MB (não alterada)
   O Rei de Amarelo (2015, Editora Clock Tower).pdf.txt 503 KB
   O Rei de Amarelo.txt 503 KB


In [11]:
from transformers import AutoTokenizer
from tqdm import tqdm
tqdm.pandas()
tok = AutoTokenizer.from_pretrained(TOKENIZER)
print('tokenizer:', TOKENIZER, '| vocab:', tok.vocab_size)
encode = lambda s: tok(s, truncation=False).input_ids
df_full['text_encoded'] = df_full.text_clean.progress_apply(lambda x: encode(x))
df_full['text_encoded_len'] = df_full.text_encoded.apply(len)
nc = int(df_full['text_clean_len'].sum()); nt = int(df_full['text_encoded_len'].sum())
print(f'chars {nc/1e6:.2f}M -> tokens {nt/1e6:.2f}M | compressão chars/tok = {nc/nt:.2f}')
print(df_full.groupby('author').agg(docs=('title','count'), toks_M=('text_encoded_len', lambda s: round(s.sum()/1e6,2))))

C:\Users\Bruno\.conda\envs\transformers-fun\Lib\site-packages\torch\cuda\__init__.py:61: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


tokenizer: artifacts/tokenizers/gpt2_ptbr_50k_v2 | vocab: 50257


  0%|          | 0/195 [00:00<?, ?it/s]

Token indices sequence length is longer than the specified maximum sequence length for this model (2144 > 1024). Running this sequence through the model will result in indexing errors


  4%|▎         | 7/195 [00:00<00:14, 12.61it/s]

...

 97%|█████████▋| 189/195 [00:33<00:01,  3.25it/s]

 98%|█████████▊| 191/195 [00:33<00:01,  3.66it/s]

 99%|█████████▉| 193/195 [00:34<00:00,  3.91it/s]

100%|██████████| 195/195 [00:34<00:00,  5.71it/s]

chars 71.47M -> tokens 15.91M | compressão chars/tok = 4.49
           docs  toks_M
author                 
chambers      1    0.12
king         65   13.38
lovecraft   114    0.66
poe           5    0.65
tolkien      10    1.09


In [12]:
df_full.to_parquet(OUT_ENC)
import os
print('salvo:', OUT_ENC, round(os.path.getsize(OUT_ENC)/1e6, 1), 'MB')
df_old = pd.read_parquet(BASE_ENC)
print()
print('== VALIDAÇÃO v21 -> v22 ==')
print('docs:', len(df_old), '->', len(df_full), f'(+{len(df_full)-len(df_old)})')
print('tokens M:', round(df_old.text_encoded_len.sum()/1e6, 2), '->', round(df_full.text_encoded_len.sum()/1e6, 2))
print('novos por autor:')
print(df_full[df_full.author==AUTOR].groupby('title').text_encoded_len.sum().sort_values(ascending=False))
print('\nPRÓXIMO PASSO: apontar o yaml para o v22')
print('  path_input_encoded:', OUT_ENC)

salvo: data/df_full_encoded_v23.pq 120.6 MB



== VALIDAÇÃO v21 -> v22 ==
docs: 194 -> 195 (+1)
tokens M: 15.79 -> 15.91
novos por autor:
title
O Rei de Amarelo (2015, Editora Clock Tower)    118909
Name: text_encoded_len, dtype: int64

PRÓXIMO PASSO: apontar o yaml para o v22
  path_input_encoded: data/df_full_encoded_v23.pq


### Como usar a v22
1. No yaml de treino (ex.: `params/v25_hermes.yaml`), trocar:
   `path_input_encoded: data/df_full_encoded_v22.pq` (e, se quiser, `name: ..._v22`).
2. Rodar o treino normalmente. O tokenizer **não** é retreinado (vocabulário já cobre o pt-BR).
3. Para o próximo autor: colocar epubs/pdfs em `data/<autor>/` (ou `data/outros/`), trocar `AUTOR` e rodar tudo de novo → v23.

Histórico: v20 → v21 (Tolkien, playground_2) → **v22 (Poe, este notebook)**.